# Drominator — Fine-tuning YOLOv11n sur dataset Gates

Pipeline : dataset synthétique sur Drive → fine-tuning YOLOv11n sur Colab T4 → artefacts vers Drive.

**Durée estimée** : 30–60 min sur T4 (50 epochs max, early stopping probable vers 25–35).

**Prérequis** :
- Runtime GPU activé : `Exécution > Modifier le type d'exécution > T4 GPU`
- Dataset uploadé sur Drive : `MyDrive/Drominator/gates.tar.gz`

In [11]:
# ── Configuration ─────────────────────────────────────────────────────────
REPO_URL           = "https://github.com/matensu/Drominator.git"  # <-- à remplir
BRANCH             = "detection-v3"
DATASET_DRIVE_PATH = "/content/drive/MyDrive/Drominator/gates.tar.gz"
OUTPUT_DRIVE_DIR   = "/content/drive/MyDrive/Drominator/runs"

EPOCHS   = 50
BATCH    = 32     # T4 16GB sweet spot pour yolo11n @ 640px
IMGSZ    = 640
WORKERS  = 4      # 2 vCPU sur T4 gratuit, 4 OK
PATIENCE = 10
DEVICE   = "0"

In [12]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise SystemExit("Pas de GPU détecté ! Active le runtime GPU dans Exécution > Modifier le type d'exécution.")

Thu Apr 30 14:19:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os, subprocess

PROJECT_DIR = "/content/Drominator"
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"{PROJECT_DIR} existe déjà — skip clone.")

os.chdir(PROJECT_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

/content/Drominator existe déjà — skip clone.


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [15]:
# Décommente pour récupérer un fix poussé pendant que le runtime tourne :
# !cd /content/Drominator && git pull

In [16]:
!pip install -q ultralytics

In [17]:
import tarfile
from pathlib import Path

DATASET_DIR = Path("/content/Drominator/datasets/gates")
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    print(f"Extraction de {DATASET_DRIVE_PATH} → {DATASET_DIR.parent}")
    with tarfile.open(DATASET_DRIVE_PATH) as tf:
        tf.extractall(DATASET_DIR.parent)
else:
    print(f"{DATASET_DIR} existe déjà — skip extraction.")

yaml_path = DATASET_DIR / "data.yaml"
if not yaml_path.exists():
    raise FileNotFoundError(f"data.yaml introuvable après extraction : {yaml_path}")

for split in ("train", "val", "test"):
    split_dir = DATASET_DIR / "images" / split
    n = sum(1 for _ in split_dir.iterdir()) if split_dir.is_dir() else 0
    print(f"  {split:<5}: {n} images")

/content/Drominator/datasets/gates existe déjà — skip extraction.
  train: 8000 images
  val  : 1000 images
  test : 500 images


In [18]:
import re
from pathlib import Path

yaml_path = Path("/content/Drominator/datasets/gates/data.yaml")
content = yaml_path.read_text()
patched = re.sub(r"^path:.*$", "path: /content/Drominator/datasets/gates", content, flags=re.MULTILINE)
yaml_path.write_text(patched)
print(yaml_path.read_text())

path: /content/Drominator/datasets/gates
train: images/train
val: images/val
test: images/test
names:
  0: gate



In [19]:
import subprocess, sys

result = subprocess.run([sys.executable, "ai/validate_gate_dataset.py"], cwd="/content/Drominator")
if result.returncode != 0:
    raise RuntimeError(f"Validation dataset échouée (exit code {result.returncode})")

## Lancement du training

In [21]:
import subprocess, sys, time
from datetime import datetime

start = time.time()
print(f"Start   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

cmd = [
    sys.executable, "ai/train_gate_detector.py",
    "--epochs",   str(EPOCHS),
    "--batch",    str(BATCH),
    "--imgsz",    str(IMGSZ),
    "--workers",  str(WORKERS),
    "--patience", str(PATIENCE),
    "--device",   DEVICE,
]
result = subprocess.run(cmd, cwd="/content/Drominator")

elapsed_min = (time.time() - start) / 60
print("=" * 60)
print(f"End     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed : {elapsed_min:.1f} min")

if result.returncode != 0:
    raise RuntimeError(f"Training échoué (exit code {result.returncode})")

Start   : 2026-04-30 15:29:05


KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/Drominator/ai/models/yolo_gate.pt")
metrics = model.val(
    data="/content/Drominator/datasets/gates/data.yaml",
    split="test",
    imgsz=IMGSZ,
    device=DEVICE,
)
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"precision: {float(metrics.box.mp):.4f}")
print(f"recall   : {float(metrics.box.mr):.4f}")

In [ ]:
import random
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

model = YOLO("/content/Drominator/ai/models/yolo_gate.pt")
test_imgs = sorted(Path("/content/Drominator/datasets/gates/images/test").glob("*.jpg"))
if not test_imgs:
    raise RuntimeError("Aucune image de test trouvée")

random.seed(42)
sample = random.sample(test_imgs, min(10, len(test_imgs)))
results = model.predict(
    [str(p) for p in sample],
    save=True,
    project="/content/Drominator/runs/detect",
    name="predict",
    exist_ok=True,
    device=DEVICE,
)
predict_dir = Path(results[0].save_dir)
print(f"Predictions saved in {predict_dir}\n")

for img in sorted(predict_dir.glob("*.jpg"))[:5]:
    display(Image(filename=str(img)))

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path(OUTPUT_DRIVE_DIR) / f"yolo_gate_{ts}"
out_dir.mkdir(parents=True, exist_ok=True)

search_roots = [
    Path("/content/Drominator/ai/models"),
    Path("/content/Drominator/runs/detect"),
]
patterns = [
    "**/results.png",
    "**/results.csv",
    "**/weights/best.pt",
    "**/weights/last.pt",
]

files_to_copy = [Path("/content/Drominator/ai/models/yolo_gate.pt")]
for root in search_roots:
    if not root.exists():
        continue
    for pattern in patterns:
        files_to_copy.extend(root.glob(pattern))

# Dédupe (yolo_gate.pt peut aussi matcher un best.pt)
files_to_copy = list(dict.fromkeys(files_to_copy))

for src in files_to_copy:
    if not src.exists():
        print(f"[WARN] absent : {src}")
        continue
    parent_name = src.parent.name
    if parent_name == "weights":
        # weights/best.pt -> grand-parent (gate_detector_warmup, gate_detector, predict, ...)
        parent_name = src.parent.parent.name
    dest_name = f"{parent_name}__{src.name}" if parent_name else src.name
    dst = out_dir / dest_name
    shutil.copy2(src, dst)
    print(f"  copié : {src} -> {dst}")

print(f"\nArtefacts dans : {out_dir}")

In [ ]:
# Décommente pour télécharger directement sur ton laptop :
# from google.colab import files
# files.download('/content/Drominator/ai/models/yolo_gate.pt')

## Training terminé

Artefacts copiés dans Drive : `OUTPUT_DRIVE_DIR/yolo_gate_<timestamp>/`.